# Preparing data for machine learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/y-rosenthal/yr-pre-ml/blob/main/jb/chapters/13-preparing-data-for-ml.ipynb)

In the chapter *A first machine-learning example*, our first `fit` failed: the data had a text column and missing
values. We fixed it with two lines that we explained only briefly. In this chapter, we prepare the same data step by
step, and we understand each step. At the end, `fit` works. The two projects that follow this chapter use exactly
these steps.

**In this chapter, you will learn to:**

- find the missing values of a DataFrame, column by column and row by row;
- drop the rows with missing values, or fill the missing values, and decide which one to do;
- turn a text column into number columns with `pd.get_dummies`;
- choose `X` and `y`, and check that `X` is ready for a model.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

DATA_URL = "https://raw.githubusercontent.com/y-rosenthal/yr-pre-ml/main/data/"

We import `LinearRegression` for the last section of the chapter.

## Why the first fit failed

We use the customers of the loyalty program again: one row for each member, and the amount they spent in a year,
`annual_spend`, which we want to predict.

In [ ]:
customers = pd.read_csv(DATA_URL + "customers.csv")
customers.head()

In that chapter, `model.fit(X, y)` stopped with an error, and the data had two problems:

1. **Text.** The column `region` holds text, such as `South`. The error said
   `could not convert string to float: 'South'`. A model computes with numbers, as in
   *spending = a + b × visits*, and it cannot multiply a number by `South`.
2. **Missing values.** Some values are missing (`NaN`, "Not a Number"). We do not see one in the first five rows,
   but there are some further down, and many of scikit-learn's models, such as `LinearRegression`, refuse them too. With the text fixed, `fit` would stop
   with a second error, `Input X contains NaN`.

We fix the missing values first, and then the text.

## Finding missing values

Let's start with a small made-up table of four new members, so that we can see every value. Python's `None`
means "no value", and pandas treats it as a missing value:

In [ ]:
members = pd.DataFrame({
    "age": [34, None, 51, 28],
    "region": ["North", "South", None, "East"],
    "visits": [4.0, 2.5, 6.1, 3.3],
})
members

pandas shows a missing number as `NaN`. In the text column `region`, it shows the missing value as `None` (in
pandas 3, as `NaN`). The method `isna()` ("is it NaN?") answers for **every** value: `True` where the value is
missing, and `False` where it is not.

In [ ]:
members.isna()

The result is a table of the same shape, full of `True` and `False`, like a mask of the chapter *Comparisons and
Boolean masks*. And as there, `sum()` counts the `True` values, because `True` counts as 1. On a DataFrame, `sum()`
gives one count for each column. Here it is for the 500 customers:

In [ ]:
customers.isna().sum()

14 ages and 11 average baskets are missing. The other columns are complete. This one line is the first thing to
run on any new data set.

## Rows with a missing value

Which customers have a missing value? `any()` asks "is at least one value `True`?". With `axis=1`, it asks this
for each row, as `axis=1` gave one result for each row in the chapter *Two-dimensional arrays*. The result is a
mask with one value for each customer, and we use it to filter the rows:

In [ ]:
missing_rows = customers.isna().any(axis=1)
customers.loc[missing_rows, ["age", "region", "visits_per_month", "avg_basket"]].head()

Each of these customers has `NaN` in `age` or in `avg_basket`. We kept the mask in the variable `missing_rows`,
because we will look at the same rows again after we fill them.

## Dropping rows with dropna()

The simplest fix is to remove every row that has a missing value. `dropna()` does that. Let's compare the shape
before and after:

In [ ]:
print(customers.shape)
print(customers.dropna().shape)
print(customers.dropna(subset=["age"]).shape)

- `customers` has 500 rows. `dropna()` returns a **new** DataFrame without the rows that have a missing value, and
  it has 475 rows: 25 customers had a missing value (14 ages and 11 baskets: no customer misses both).
  `customers` itself does not change.
- `dropna(subset=["age"])` looks only at the columns in the list. It drops the 14 rows without an age, and keeps
  the 11 rows without a basket: 486 rows remain. This is useful when we need only some of the columns.

The first example used `dropna()`, and lost 5% of the customers. Next, we see how to keep them.

## Filling missing numbers with fillna()

Instead of dropping a customer, we can fill in a value for the missing one. `fillna(value)` returns a copy of a
column with every missing value replaced by `value`.

Which value? For a number column, a common choice is the **median**, the middle value when the values are sorted
(we met `np.median` in the chapter *Comparisons and Boolean masks*). Unlike the average, the median does not move
much when a few values are very large. We fill both columns with their medians, and look at the same rows again:

In [ ]:
customers["age"] = customers["age"].fillna(customers["age"].median())
customers["avg_basket"] = customers["avg_basket"].fillna(customers["avg_basket"].median())
customers.loc[missing_rows, ["age", "region", "visits_per_month", "avg_basket"]].head()

Each line computes the median of the column, fills the missing values with it, and puts the result back into the
column with `=`. The four customers here who had no age now have the age 46.0, the median age. The last customer had
no basket amount, and now has 51.56, the median basket. All 500 customers are still there.

> **Common mistake:** writing `customers["age"].fillna(46)` on a line of its own. `fillna` returns a new column
> and does not change `customers`, so nothing is filled. Always assign the result back:
> `customers["age"] = customers["age"].fillna(46)`. (You may see older code that writes `inplace=True` instead.
> Do not use it: in pandas 3, `customers["age"].fillna(46, inplace=True)` does not change `customers` at all.)

## Filling missing text with "Unknown"

`customers.csv` has no missing text, so we go back to the small table. There is no median of text. A simple and
honest choice is a new value, `"Unknown"`: it keeps the row, and it says that we do not know.

In [ ]:
members["region"] = members["region"].fillna("Unknown")
members["age"] = members["age"].fillna(members["age"].median())
members

The third member's region is now `Unknown`, and the second member's age is 34.0, the median of 34, 51 and 28.
"Unknown" becomes a category of its own. Members with an unknown region may differ from the others, and a model
can learn that.

## When to drop and when to fill

Both ways have a cost. Dropping a row throws away everything else that we know about that customer: a customer
without an age still has a region, visits and a basket. Filling keeps the customer, but puts in a value that we
made up. The median is a reasonable guess, not the truth.

Some rules of thumb:

- **Few rows affected: dropping is fine.** Here, only 25 of 500 rows (5%) have a missing value.
- **Many rows would go: fill.** One missing value should not cost us a whole row of useful facts.
- **New data will have gaps too: fill.** A model that predicts for new customers meets missing values again. We
  can fill a new customer's missing age with the same median, but we cannot drop a customer that we must predict.
- **Never fill the target.** If `annual_spend` were missing for a customer, drop that row. A made-up target would
  teach the model a made-up answer.
- Fill a number column with its median, and a text column with `"Unknown"`.

Whatever we choose, we say so when we report the results. In this chapter, we filled.

## Text to numbers with get_dummies()

Now the text. The rule is simple: for a text column, make one new column for each of its values. In each new
column, a row gets 1 if it has that value, and 0 if not. Such 0/1 columns are called **dummy variables**, and
`pd.get_dummies` makes them. The small table first:

In [ ]:
pd.get_dummies(members, columns=["region"], dtype=int)

The column `region` is gone. In its place are four columns, one for each value: `region_East`, `region_North`,
`region_South` and `region_Unknown`. Each row has exactly one 1: the first member is from the North, so it has a 1
in `region_North`. `dtype=int` asks for the numbers 0 and 1. Without it, the new columns hold `True` and `False`,
as in the first example. A model accepts both, but 0 and 1 are easier to read.

`get_dummies` returns a new DataFrame, so we assign it back, as with `fillna`. Now the customers:

In [ ]:
customers = pd.get_dummies(customers, columns=["region"], dtype=int)
customers.columns

The four regions became four columns at the end, and `region` is gone. `customers` now has 9 columns, all of them
numbers. (The `dtype` at the bottom of the list of names is `object`, or `str` in pandas 3: the column *names* are
text, which is fine.)

## Choosing X and y

The predictors `X` are all the columns except the target, and the target `y` is one column:

In [ ]:
X = customers.drop(columns="annual_spend")
y = customers["annual_spend"]
X.shape, y.shape

`X` is a table with 500 rows and 8 columns, and `y` is one column with 500 values. The numbers of rows must be
equal (`X.shape[0] == y.shape[0]`): row 0 of `X` describes the customer whose spending is value 0 of `y`.

> **Common mistake:** leaving the target in `X`. If `annual_spend` stays among the predictors, the model just
> copies it. It then looks perfect on our data, and is useless for a new customer, whose spending we do not know.
> `drop(columns="annual_spend")` removes it.

## Checking that X is ready

Before we call `fit`, we check two things: no value is missing, and every column holds numbers.

In [ ]:
print(X.isna().sum().sum() == 0)
X.dtypes

`X.isna().sum()` gives one count of missing values for each column, and the second `.sum()` adds the counts up.
The total is 0, so the comparison prints `True`. The dtypes are all `int64` (whole numbers) or `float64` (numbers
with decimals). No column is `object` (or `str` in pandas 3), so no text is left.

## The NumPy array inside

A model does not work on a DataFrame directly. `fit` turns `X` into a two-dimensional NumPy array, as in the
chapter *Two-dimensional arrays*. `to_numpy()` shows us that array:

In [ ]:
X_array = X.to_numpy()
X_array[:3]

Each row of the array is a customer, and each column a predictor, in the order of `X.columns`: the first customer
is 29 years old, a customer for 8 months, and so on, and the `1.` in the seventh column says that this customer is
in the South. All the values are floats, because an array holds one type, and mixing whole numbers and decimals
gives floats. That is why every column must be a number: an array of numbers cannot hold the text `South` or a gap.

## Now the model fits

Everything is ready. This is the line that failed in the first example:

In [ ]:
model = LinearRegression()
model.fit(X, y)

No error. Under the cell, Colab shows a short description of the model: `fit` worked, and the model has learned
its numbers from the 500 customers. The next steps (a training set and a test set, predictions,
the mean absolute error) are the model code of the chapter *A first machine-learning example*, and the two projects
use it.

## Summary

The steps that make a DataFrame ready for a model:

1. **Find the missing values:** `df.isna().sum()` counts them in each column, and `df.isna().any(axis=1)` gives a
   mask of the rows that have one.
2. **Drop or fill them:** `df = df.dropna()` (or `dropna(subset=[...])`) drops rows.
   `df["col"] = df["col"].fillna(df["col"].median())` fills a number column, and
   `df["col"] = df["col"].fillna("Unknown")` a text column. Drop when few rows are affected, fill when many would
   go. Never fill the target.
3. **Turn text into numbers:** `df = pd.get_dummies(df, columns=["col"], dtype=int)` makes one 0/1 column for
   each value.
4. **Choose `X` and `y`:** `X = df.drop(columns="target")` and `y = df["target"]`.
5. **Check `X`:** `X.isna().sum().sum() == 0`, `X.dtypes` (numbers only), and `X.shape[0] == y.shape[0]`.

Then `fit` works. `X.to_numpy()` shows the NumPy array that the model gets.

## Further reading

- The pandas guide "Working with missing data" (<https://pandas.pydata.org/docs/user_guide/missing_data.html>)
  shows more ways to find and fill missing values. Some of it goes beyond this book.

## Practice questions

### Question 1 (Easy)

Read `customers.csv` again into a new variable, `raw`. How many values are missing in the whole table?

<details class="yr-answer">
<summary>Answer</summary>

`isna().sum()` counts the missing values of each column, and the second `sum()` adds these counts up: 25.

```python
raw = pd.read_csv(DATA_URL + "customers.csv")
print(raw.isna().sum().sum())
```

</details>

In [ ]:
# Write your answer here

### Question 2 (Easy)

Why must `annual_spend` not be one of the columns of `X`?

<details class="yr-answer">
<summary>Answer</summary>

`annual_spend` is the target, the value that we want to predict. If it is in `X`, the model can copy it, and it
looks perfect. But for a new customer, we do not know the spending: that is why we predict it. So the model must
learn from the other columns only.

</details>

In [ ]:
# Write your answer here

### Question 3 (Easy)

A table of four orders has a column `payment` with the values `Card`, `Cash`, `Card` and `Phone`. How many dummy
columns does `pd.get_dummies` make, and what are their names? Check with code.

<details class="yr-answer">
<summary>Answer</summary>

One column for each different value: three columns, `payment_Card`, `payment_Cash` and `payment_Phone`.

```python
orders = pd.DataFrame({"payment": ["Card", "Cash", "Card", "Phone"]})
pd.get_dummies(orders, columns=["payment"], dtype=int)
```

</details>

In [ ]:
# Write your answer here

### Question 4 (Medium)

Five customers have the average baskets 20, 25, 30, 35 and 400 dollars. Compute the average and the median. Which
one is the better value to fill in for a missing basket, and why?

<details class="yr-answer">
<summary>Answer</summary>

The average is 102 and the median is 30. The one very large basket pulls the average far above four of the five
customers. The median, 30, is much closer to a typical customer, so it is the better guess.

```python
baskets = pd.Series([20, 25, 30, 35, 400])
print(baskets.mean())
print(baskets.median())
```

</details>

In [ ]:
# Write your answer here

### Question 5 (Medium)

Start from `raw = pd.read_csv(DATA_URL + "customers.csv")`. Drop only the rows where `avg_basket` is missing. How
many rows remain? Which column still has missing values?

<details class="yr-answer">
<summary>Answer</summary>

489 rows remain (500 minus the 11 rows without a basket). `age` still has its 14 missing values, because
`subset` looks only at `avg_basket`.

```python
raw = pd.read_csv(DATA_URL + "customers.csv")
some = raw.dropna(subset=["avg_basket"])
print(some.shape)
print(some.isna().sum())
```

</details>

In [ ]:
# Write your answer here

### Question 6 (Medium)

Here is a small table of deliveries. Fill the missing `minutes` with the median, and the missing `weather` with
`"Unknown"`. Then replace `weather` with 0/1 dummy columns.

```python
trips = pd.DataFrame({
    "minutes": [25.0, None, 40.0, 31.0],
    "weather": ["Clear", "Rain", None, "Clear"],
})
```

<details class="yr-answer">
<summary>Answer</summary>

The median of 25, 40 and 31 is 31, so the second delivery gets 31.0 minutes. The weather becomes three columns:
`weather_Clear`, `weather_Rain` and `weather_Unknown`.

```python
trips = pd.DataFrame({
    "minutes": [25.0, None, 40.0, 31.0],
    "weather": ["Clear", "Rain", None, "Clear"],
})
trips["minutes"] = trips["minutes"].fillna(trips["minutes"].median())
trips["weather"] = trips["weather"].fillna("Unknown")
trips = pd.get_dummies(trips, columns=["weather"], dtype=int)
trips
```

</details>

In [ ]:
# Write your answer here

### Question 7 (Hard)

Prepare `customers.csv` again, but this time **drop** the rows with missing values instead of filling them. Make
the dummies, choose `X` and `y`, print their shapes, and fit a `LinearRegression`.

<details class="yr-answer">
<summary>Answer</summary>

After `dropna()`, 475 customers remain, so `X` has the shape (475, 8) and `y` the shape (475,). `fit` works,
because nothing is missing and every column is a number.

```python
dropped = pd.read_csv(DATA_URL + "customers.csv")
dropped = dropped.dropna()
dropped = pd.get_dummies(dropped, columns=["region"], dtype=int)
X_dropped = dropped.drop(columns="annual_spend")
y_dropped = dropped["annual_spend"]
print(X_dropped.shape, y_dropped.shape)
model_dropped = LinearRegression()
model_dropped.fit(X_dropped, y_dropped)
```

</details>

In [ ]:
# Write your answer here

### Question 8 (Challenge)

Write a function `prepare(df)` that does all the steps of this chapter on a freshly read `customers.csv`
(fill both number columns with their medians, make the region dummies, choose `X` and `y`) and returns `X` and `y`.
Test it, and check that the result has no missing value.

<details class="yr-answer">
<summary>Answer</summary>

A function can return two values, and we unpack them as `X2, y2 = ...`. `df.copy()` makes the function work on a
copy, so that it does not change the DataFrame that it gets. The shapes are (500, 8) and (500,), and the check
prints `True`.

```python
def prepare(df):
    df = df.copy()
    df["age"] = df["age"].fillna(df["age"].median())
    df["avg_basket"] = df["avg_basket"].fillna(df["avg_basket"].median())
    df = pd.get_dummies(df, columns=["region"], dtype=int)
    X = df.drop(columns="annual_spend")
    y = df["annual_spend"]
    return X, y


X2, y2 = prepare(pd.read_csv(DATA_URL + "customers.csv"))
print(X2.shape, y2.shape)
print(X2.isna().sum().sum() == 0)
```

</details>

In [ ]:
# Write your answer here